In [ ]:
# exp01_all_noise_agent1_vs_baselines_mae_vs_nl_ijcai_singlecol_legend_bottom.py
# Single-column (IJCAI) friendly:
#   - keep 1x3 layout
#   - increase height
#   - move legend to bottom (NO overlap with panels)
#   - manual spacing via subplots_adjust (predictable)
#
# Run:
#   python exp01_all_noise_agent1_vs_baselines_mae_vs_nl_ijcai_singlecol_legend_bottom.py

import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl")

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib
matplotlib.use("Agg")  # safe for WSL/CLI
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator, MaxNLocator


# -------- paths (edit if needed) --------
PATHS = {
    "iq_chain": "exp01_ibm_baseline+agent__iq_chain__mean_abs_err_MHz(1).csv",
    "chain": "exp01_ibm_baseline+agent__chain__mean_abs_err_MHz(1).csv",
    "realistic_chain": "exp01_ibm_baseline+agent__realistic_chain__mean_abs_err_MHz(1).csv",
}

OUT_PNG = "exp01_all_noise_agent1_vs_baselines_mae_vs_nl_ijcai_singlecol_bottomlegend.png"
OUT_PDF = "exp01_all_noise_agent1_vs_baselines_mae_vs_nl_ijcai_singlecol_bottomlegend.pdf"


def load_table(csv_path: str):
    """Load one table: first col is method name; remaining cols are noise levels (nl)."""
    df = pd.read_csv(csv_path)
    id_col = df.columns[0]
    x_cols = [c for c in df.columns if c != id_col]

    xs = np.array(sorted([float(c) for c in x_cols]))
    col_map = {float(c): c for c in x_cols}
    x_cols_ordered = [col_map[x] for x in xs]

    labels = df[id_col].astype(str).tolist()
    series = {
        lab: df.loc[i, x_cols_ordered].astype(float).to_numpy()
        for i, lab in enumerate(labels)
    }
    return xs, labels, series


def pick_agent_label(labels):
    """Robustly find Agent1 row name if variant exists."""
    for cand in labels:
        if cand.lower() in {"agent1", "a1", "onlyagent", "agent"}:
            return cand
    return "Agent1" if "Agent1" in labels else None


def main():
    # IJCAI-like styling (slightly heavier for readability in small figures)
    mpl.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset": "dejavuserif",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.9,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.major.width": 0.9,
        "ytick.major.width": 0.9,
        "xtick.minor.width": 0.7,
        "ytick.minor.width": 0.7,
    })

    # Load all three
    tables = {k: load_table(v) for k, v in PATHS.items()}

    # Global y-limit and x-range
    global_ymax = 0.0
    x_min, x_max = None, None
    for _, (xs, _, series) in tables.items():
        global_ymax = max(global_ymax, max(float(np.nanmax(y)) for y in series.values()))
        x_min = float(np.min(xs)) if x_min is None else min(x_min, float(np.min(xs)))
        x_max = float(np.max(xs)) if x_max is None else max(x_max, float(np.max(xs)))

    # Stable marker assignment across panels
    all_labels = []
    for _, (_, labels, _) in tables.items():
        for l in labels:
            if l not in all_labels:
                all_labels.append(l)

    marker_cycle = ["o", "s", "^", "v", "P", "X", "D", "<", ">"]
    mk = {lab: marker_cycle[i % len(marker_cycle)] for i, lab in enumerate(all_labels)}
    mk["Agent1"] = "D"

    # ---- single-column width, increased height ----
    # Single column ~3.25in width. Use 3.35in and increase height for readability.
    fig, axes = plt.subplots(1, 3, figsize=(3.35, 2.65), sharey=True)

    panel_order = ["iq_chain", "chain", "realistic_chain"]
    panel_titles = {"iq_chain": "iq_chain", "chain": "chain", "realistic_chain": "realistic_chain"}
    panel_letters = ["(a)", "(b)", "(c)"]

    # Collect handles for a shared legend
    handles_for_legend = {}

    # visual knobs
    label_fs = 9
    tick_fs = 8
    title_fs = 9
    legend_fs = 7.5

    baseline_lw = 1.2
    baseline_ms = 3.4
    baseline_alpha = 0.75

    agent_lw = 2.2
    agent_ms = 4.0

    for ax, key, letter in zip(axes, panel_order, panel_letters):
        xs, labels, series = tables[key]
        agent_label = pick_agent_label(labels)
        baseline_labels = [l for l in labels if l != agent_label]

        # Baselines
        for lab in baseline_labels:
            line, = ax.plot(
                xs, series[lab],
                marker=mk.get(lab, "o"),
                linewidth=baseline_lw,
                markersize=baseline_ms,
                alpha=baseline_alpha,
                label=lab
            )
            if lab not in handles_for_legend:
                handles_for_legend[lab] = line

        # Agent emphasized
        if agent_label is not None:
            line, = ax.plot(
                xs, series[agent_label],
                marker=mk.get("Agent1", "D"),
                linewidth=agent_lw,
                markersize=agent_ms,
                label="Agent1"
            )
            if "Agent1" not in handles_for_legend:
                handles_for_legend["Agent1"] = line

        # Title inside (saves space)
        ax.text(
            0.02, 0.98, f"{letter} {panel_titles[key]}",
            transform=ax.transAxes, ha="left", va="top",
            fontsize=title_fs
        )

        # Ticks: keep sparse (panels are narrow)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
        ax.xaxis.set_minor_locator(AutoMinorLocator(2))
        ax.yaxis.set_minor_locator(AutoMinorLocator(2))

        ax.tick_params(axis="both", which="major", length=3.0, pad=1.0, labelsize=tick_fs)
        ax.tick_params(axis="both", which="minor", length=1.7)

        ax.grid(True, which="major", linestyle="--", linewidth=0.55, alpha=0.30)
        ax.grid(True, which="minor", linestyle=":", linewidth=0.45, alpha=0.22)

        ax.set_xlim(x_min, x_max)
        ax.set_ylim(0, global_ymax * 1.05)

        ax.set_xlabel("nl", fontsize=label_fs, labelpad=1)

    axes[0].set_ylabel("MAE", fontsize=label_fs, labelpad=2)

    # ---- Legend at bottom (NO overlap) ----
    legend_labels = list(handles_for_legend.keys())
    if "Agent1" in legend_labels:
        legend_labels.remove("Agent1")
        legend_labels = ["Agent1"] + sorted(legend_labels)
    else:
        legend_labels = sorted(legend_labels)

    legend_handles = [handles_for_legend[l] for l in legend_labels]

    # Choose number of columns: 2 is safer for long method names in single-column.
    # If your method names are short, you can try ncol=3.
    leg = fig.legend(
        legend_handles, legend_labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.02),
        ncol=2,
        fontsize=legend_fs,
        frameon=True,
        fancybox=False,
        framealpha=0.95,
        borderpad=0.30,
        handlelength=1.8,
        handletextpad=0.5,
        columnspacing=0.9,
        labelspacing=0.35
    )
    leg.get_frame().set_linewidth(0.7)

    # Manual spacing: reserve bottom space for legend; control panel gaps with wspace
    fig.subplots_adjust(
        left=0.12, right=0.98, top=0.96, bottom=0.30,
        wspace=0.28
    )

    fig.savefig(OUT_PNG, dpi=600, bbox_inches="tight")
    fig.savefig(OUT_PDF, bbox_inches="tight")
    plt.close(fig)

    print("Saved:")
    print(" -", OUT_PNG)
    print(" -", OUT_PDF)


if __name__ == "__main__":
    main()
